# 01 — EDA AirBnB · P8-Data-Analyst
**Bootcamp IA P4 · Factoría F5 Madrid**  
**Python:** 3.13 · **Librerías:** pandas, numpy, matplotlib, seaborn, plotly  
**Repositorio:** https://github.com/Jose-JulioRamirezySanchez-Escobar/P8-Data-Analyst

---

## Objetivo de este notebook

Realizar el **Análisis Exploratorio de Datos (EDA)** completo sobre los datasets de AirBnB de 6 ciudades:  
Madrid, Londres, Milán, Nueva York, Sydney y Tokio.

El EDA es el primer paso obligatorio de cualquier proyecto de datos. Su propósito es:
- Conocer la estructura y calidad de los datos antes de transformarlos
- Detectar problemas (nulos, outliers, inconsistencias) que afecten al análisis
- Generar hipótesis sobre patrones y relaciones entre variables
- Documentar decisiones para que el equipo pueda reproducir y validar el trabajo

> **[SDD — Spec-Driven Development]**  
> Cada sección de este notebook corresponde a una especificación del proyecto.  
> Los comentarios `# SDD:` en el código indican qué criterio de la rúbrica cubre cada bloque.  
> VSCode puede usar estas especificaciones para generar o completar código automáticamente.

## 0. Imports y configuración

Cargamos todas las librerías necesarias al inicio del notebook.  
Esto es una buena práctica porque:
- Hace explícitas las dependencias del notebook
- Permite detectar librerías no instaladas antes de ejecutar el análisis
- Facilita la revisión del código por otros miembros del equipo

In [ ]:
# SDD: Imports base del proyecto
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import warnings

warnings.filterwarnings('ignore')

# Configuración visual global
sns.set_theme(style='whitegrid', palette='tab10')
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['figure.dpi'] = 100

print('✅ Librerías cargadas correctamente')
print(f'   pandas  {pd.__version__}')
print(f'   numpy   {np.__version__}')
print(f'   seaborn {sns.__version__}')

---
## 1. Carga de datos

Cargamos los 6 datasets de ciudades. Usamos las versiones más recientes disponibles:  
- **Madrid, Londres, Milán:** archivos `.xlsx` (2024-2026)
- **Nueva York, Sydney, Tokio:** archivos `.csv` (únicas versiones disponibles)

### ¿Por qué añadimos la columna `city`?

Para poder comparar ciudades en un único DataFrame combinado necesitamos una columna que identifique el origen de cada fila. Sin esta columna, al hacer `pd.concat()` perderíamos el contexto de a qué ciudad pertenece cada alojamiento.

### ¿Por qué usamos solo las columnas comunes?

No todos los datasets tienen las mismas columnas (Tokio no tiene `availability_365` ni `calculated_host_listings_count`; Milán no tiene `neighbourhood_group`). Para el análisis comparativo usamos el subconjunto común. Los análisis específicos por ciudad pueden usar todas las columnas disponibles.

In [ ]:
# SDD: Carga de datasets — Rúbrica competencia 5: Uso y gestión de formato .csv/.xlsx

BASE_URL = 'https://raw.githubusercontent.com/Jose-JulioRamirezySanchez-Escobar/P8-Data-Analyst/main/data/raw/'

# CSVs históricos
df_ny     = pd.read_csv(BASE_URL + 'NY_airbnb.csv')
df_sydney = pd.read_csv(BASE_URL + 'sydney_airbnb.csv')
df_tokyo  = pd.read_csv(BASE_URL + 'tokyo_airbnb.csv')
df_london_csv = pd.read_csv(BASE_URL + 'london_airbnb.csv')
df_madrid_csv = pd.read_csv(BASE_URL + 'madrid_airbnb.csv')
df_milan_csv  = pd.read_csv(BASE_URL + 'milan_airbnb.csv')

# XLSX más recientes — requiere openpyxl
# Para cargar desde URL necesitamos io.BytesIO
import requests
from io import BytesIO

def load_xlsx(url):
    r = requests.get(url)
    return pd.read_excel(BytesIO(r.content), engine='openpyxl')

df_madrid  = load_xlsx(BASE_URL + 'madrid_airbnb.xlsx')
df_london  = load_xlsx(BASE_URL + 'london_airbnb.xlsx')
df_milan   = load_xlsx(BASE_URL + 'milan_airbnb.xlsx')

# Añadir columna ciudad a los CSV históricos
df_ny['city']         = 'New York'
df_sydney['city']     = 'Sydney'
df_tokyo['city']      = 'Tokyo'
df_london_csv['city'] = 'London'
df_madrid_csv['city'] = 'Madrid'
df_milan_csv['city']  = 'Milan'

# Dataset combinado con columnas comunes
COLS_COMUNES = [
    'id', 'name', 'host_id', 'host_name', 'neighbourhood',
    'latitude', 'longitude', 'room_type', 'price',
    'minimum_nights', 'number_of_reviews',
    'last_review', 'reviews_per_month', 'city'
]

dfs = [df_ny, df_madrid_csv, df_tokyo, df_sydney, df_london_csv, df_milan_csv]
df_all = pd.concat([df[COLS_COMUNES] for df in dfs], ignore_index=True)

print(f'Dataset combinado: {df_all.shape[0]:,} filas × {df_all.shape[1]} columnas')
print(f'Ciudades: {df_all["city"].unique()}')
df_all.head(3)

---
## 2. Estadística descriptiva

La estadística descriptiva nos da una **visión global del dataset** antes de entrar en análisis detallados.

### `df.describe()` — ¿por qué?
Calcula automáticamente: count, media, desviación estándar, mínimo, percentiles 25/50/75 y máximo para todas las variables numéricas. Permite detectar de un vistazo:
- Rangos anómalos (precio = 0, precio = 99999)
- Variables con alta dispersión (desviación estándar muy alta vs. media)
- Diferencias entre media y mediana que sugieren distribuciones sesgadas

### `df.info()` — ¿por qué?
Muestra el tipo de dato de cada columna y el número de valores no nulos. Es fundamental para:
- Detectar columnas con nulos (número de non-null < total filas)
- Identificar columnas con tipo incorrecto (precio como object en lugar de float)

### `value_counts()` — ¿por qué?
Para variables categóricas, muestra la distribución de frecuencias. Permite detectar:
- Categorías dominantes (¿hay un tipo de alojamiento mucho más frecuente?)
- Posibles errores de escritura ('Entire home/apt' vs 'entire home/apt')
- Clases desbalanceadas relevantes para ML posterior

In [ ]:
# SDD: Estadística descriptiva — Rúbrica competencia 5: Análisis exploratorio detallado (EDA)
print('=== SHAPE ===')
print(f'Filas: {df_all.shape[0]:,} | Columnas: {df_all.shape[1]}')

print('\n=== TIPOS Y NULOS ===')
df_all.info()

In [ ]:
print('=== ESTADÍSTICA DESCRIPTIVA (numéricas) ===')
df_all.describe().round(2)

In [ ]:
print('=== FRECUENCIAS CATEGÓRICAS ===')
print('\n-- room_type --')
print(df_all['room_type'].value_counts())
print('\n-- city --')
print(df_all['city'].value_counts())

In [ ]:
# Resumen de nulos por columna
nulos = df_all.isnull().sum()
nulos_pct = (nulos / len(df_all) * 100).round(2)
pd.DataFrame({'nulos': nulos, '%': nulos_pct}).query('nulos > 0').sort_values('%', ascending=False)

---
## 3. Distribución de variables

### Histogramas — ¿por qué?
Un histograma muestra cómo se distribuyen los valores de una variable continua dividiendo el rango en intervalos (bins) y contando cuántas observaciones caen en cada uno. Es la herramienta básica para entender si una variable sigue una distribución normal, está sesgada, o tiene múltiples modas. En el contexto de precios de AirBnB esperamos una distribución muy sesgada a la derecha (pocos alojamientos muy caros).

### Boxplots — ¿por qué?
El boxplot resume la distribución en 5 estadísticos: mínimo, Q1, mediana, Q3 y máximo. Los puntos fuera de los bigotes son outliers según el criterio IQR. Es especialmente útil para **comparar distribuciones entre grupos** (precio por ciudad, precio por tipo de habitación) de forma compacta.

### Gráficos de barras — ¿por qué?
Para variables categóricas (room_type, city) el gráfico de barras es la representación natural de las frecuencias. Permite ver de inmediato qué categorías dominan el dataset.

In [ ]:
# SDD: Distribución de variables — Rúbrica: Visualización de datos (seaborn, matplotlib)

# Filtro para visualización: excluir precios extremos (top 1%)
PRECIO_MAX_VIZ = df_all['price'].quantile(0.99)
df_viz = df_all[df_all['price'] <= PRECIO_MAX_VIZ].copy()

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Histograma de precios por ciudad
sns.histplot(data=df_viz, x='price', hue='city', bins=60, ax=axes[0], alpha=0.6)
axes[0].set_title('Distribución de precios por ciudad')
axes[0].set_xlabel('Precio (moneda local)')

# Histograma de número de reviews
sns.histplot(data=df_viz[df_viz['number_of_reviews'] < 200],
             x='number_of_reviews', bins=50, ax=axes[1], color='steelblue')
axes[1].set_title('Distribución de número de reviews')

plt.tight_layout()
plt.show()

In [ ]:
# Boxplot de precio por tipo de alojamiento
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

sns.boxplot(data=df_viz, x='room_type', y='price', ax=axes[0], palette='tab10')
axes[0].set_title('Precio por tipo de alojamiento')
axes[0].tick_params(axis='x', rotation=15)

sns.boxplot(data=df_viz, x='city', y='price', ax=axes[1], palette='tab10')
axes[1].set_title('Precio por ciudad')
axes[1].tick_params(axis='x', rotation=15)

plt.tight_layout()
plt.show()

In [ ]:
# Gráfico de barras: frecuencia de tipos de alojamiento
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

sns.countplot(data=df_all, x='room_type',
              order=df_all['room_type'].value_counts().index, ax=axes[0], palette='tab10')
axes[0].set_title('Frecuencia por tipo de alojamiento (global)')
axes[0].tick_params(axis='x', rotation=15)

sns.countplot(data=df_all, x='city',
              order=df_all['city'].value_counts().index, ax=axes[1], palette='tab10')
axes[1].set_title('Número de alojamientos por ciudad')

plt.tight_layout()
plt.show()

---
## 4. Detección de outliers

### Método IQR — ¿por qué?

El método del **rango intercuartílico (IQR)** es el estándar para detectar outliers en distribuciones no normales (como precios, que siempre están sesgados a la derecha).

**Fórmula:**
- IQR = Q3 − Q1  
- Límite inferior = Q1 − 1.5 × IQR  
- Límite superior = Q3 + 1.5 × IQR  
- Todo valor fuera de este rango se considera outlier

**¿Por qué 1.5?** Es la convención estadística de John Tukey (creador del boxplot). Con una distribución normal capta aprox. el 0.7% más extremo. Para distribuciones más sesgadas como los precios, puede ser conveniente usar 3×IQR para ser más conservador.

### ¿Qué hacer con los outliers?
Las opciones son:
1. **Eliminar:** cuando son errores claros (precio = 0, precio = 99999)
2. **Imputar:** sustituir por la mediana o un valor más razonable
3. **Mantener:** si el outlier es real y tiene significado de negocio (un penthouse de lujo puede costar 10× la media)

> **La decisión debe documentarse explícitamente con justificación.**

In [ ]:
# SDD: Detección de outliers — Rúbrica: Técnicas avanzadas de limpieza de datos

def detectar_outliers_iqr(series: pd.Series, k: float = 1.5) -> dict:
    """Devuelve estadísticos IQR y máscara de outliers."""
    Q1 = series.quantile(0.25)
    Q3 = series.quantile(0.75)
    IQR = Q3 - Q1
    lim_inf = Q1 - k * IQR
    lim_sup = Q3 + k * IQR
    outliers = series[(series < lim_inf) | (series > lim_sup)]
    return {
        'Q1': Q1, 'Q3': Q3, 'IQR': IQR,
        'lim_inf': lim_inf, 'lim_sup': lim_sup,
        'n_outliers': len(outliers),
        'pct_outliers': round(len(outliers) / len(series) * 100, 2)
    }

# Análisis de outliers en precio
stats_precio = detectar_outliers_iqr(df_all['price'].dropna())
print('=== OUTLIERS EN PRECIO (IQR k=1.5) ===')
for k, v in stats_precio.items():
    print(f'  {k}: {v}')

In [ ]:
# Visualización de outliers por ciudad
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for i, city in enumerate(df_all['city'].unique()):
    df_city = df_all[df_all['city'] == city]
    sns.boxplot(y=df_city['price'], ax=axes[i], color='steelblue')
    st = detectar_outliers_iqr(df_city['price'].dropna())
    axes[i].set_title(f'{city}\nOutliers: {st["n_outliers"]} ({st["pct_outliers"]}%)')
    axes[i].set_ylabel('Precio (moneda local)')

plt.suptitle('Detección de outliers por ciudad (método IQR)', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# === DECISIÓN DOCUMENTADA ===
# Los outliers de precio (IQR k=1.5) representan alojamientos de lujo reales.
# Decisión: usar k=3.0 para el dataset limpio (más conservador).
# Se eliminan solo precios = 0 (errores evidentes) y extremos absolutos.

stats_k3 = detectar_outliers_iqr(df_all['price'].dropna(), k=3.0)
lim_sup = stats_k3['lim_sup']

df_clean = df_all[
    (df_all['price'] > 0) &
    (df_all['price'] <= lim_sup)
].copy()

print(f'Dataset original: {len(df_all):,} filas')
print(f'Dataset limpio:   {len(df_clean):,} filas')
print(f'Eliminadas:       {len(df_all) - len(df_clean):,} filas ({(len(df_all)-len(df_clean))/len(df_all)*100:.1f}%)')

---
## 5. Correlaciones

### Matriz de correlación — ¿por qué?

La correlación mide la **fuerza y dirección de la relación lineal** entre dos variables numéricas. Valores entre -1 y 1:
- 1 → correlación positiva perfecta
- 0 → sin correlación lineal
- -1 → correlación negativa perfecta

Es especialmente útil para:
1. **Feature selection:** variables muy correlacionadas entre sí aportan información redundante al modelo
2. **Hipótesis:** ¿a más reviews, menor precio? ¿más noches mínimas, menos disponibilidad?
3. **Detección de multicolinealidad:** problema en modelos de regresión

### Heatmap — ¿por qué?

El heatmap codifica los valores de la matriz en color, lo que permite identificar patrones de correlación de forma visual e inmediata. Con `annot=True` muestra los valores numéricos dentro de cada celda.

In [ ]:
# SDD: Correlaciones — Rúbrica: Análisis exploratorio detallado (EDA)

COLS_NUMERICAS = ['price', 'minimum_nights', 'number_of_reviews', 'reviews_per_month']

corr_matrix = df_clean[COLS_NUMERICAS].corr()

plt.figure(figsize=(8, 6))
sns.heatmap(
    corr_matrix,
    annot=True, fmt='.2f',
    cmap='coolwarm', center=0,
    square=True, linewidths=0.5
)
plt.title('Matriz de correlación — Variables numéricas')
plt.tight_layout()
plt.show()

print('\nCorrelaciones con precio:')
print(corr_matrix['price'].sort_values(ascending=False))

In [ ]:
# Pairplot para ver relaciones bivariadas
# NOTA: ejecutar con muestra para no saturar memoria
df_sample = df_clean.sample(min(3000, len(df_clean)), random_state=42)

sns.pairplot(
    df_sample[COLS_NUMERICAS + ['city']],
    hue='city', plot_kws={'alpha': 0.3}, diag_kind='hist'
)
plt.suptitle('Pairplot — Relaciones entre variables numéricas', y=1.02)
plt.show()

---
## 6. Patrones generales

### Distribución geográfica — ¿por qué?

Los datasets incluyen coordenadas (`latitude`, `longitude`) para cada alojamiento. Visualizarlos en un mapa permite:
- Detectar concentraciones geográficas (zonas turísticas, centros urbanos)
- Identificar outliers geográficos (coordenadas en el mar, fuera de la ciudad)
- Relacionar precio con zona geográfica

### Relación precio vs. otras variables — ¿por qué?

El precio es la variable más relevante desde el punto de vista de negocio (AirBnB, inversores, propietarios). Entender qué variables lo explican es el núcleo del análisis.

In [ ]:
# SDD: Patrones generales — Rúbrica: Visualización de datos (plotly)

# Mapa geográfico interactivo
df_map = df_clean.dropna(subset=['latitude', 'longitude']).sample(min(5000, len(df_clean)), random_state=42)

fig = px.scatter_mapbox(
    df_map,
    lat='latitude', lon='longitude',
    color='city',
    size='price',
    hover_name='name',
    hover_data={'price': True, 'room_type': True, 'neighbourhood': True},
    mapbox_style='open-street-map',
    zoom=2,
    title='Distribución geográfica de alojamientos AirBnB'
)
fig.show()

In [ ]:
# Precio medio por ciudad y tipo de alojamiento
precio_pivot = df_clean.groupby(['city', 'room_type'])['price'].median().reset_index()

fig = px.bar(
    precio_pivot, x='city', y='price', color='room_type',
    barmode='group',
    title='Precio mediano por ciudad y tipo de alojamiento',
    labels={'price': 'Precio mediano (moneda local)', 'city': 'Ciudad'}
)
fig.show()

In [ ]:
# Precio vs. número de reviews (relación)
fig = px.scatter(
    df_clean.sample(min(3000, len(df_clean)), random_state=42),
    x='number_of_reviews', y='price',
    color='city', opacity=0.5,
    trendline='ols',
    title='Relación precio vs. número de reviews'
)
fig.show()

---
## 7. Limpieza y preprocesado

Aplicamos las transformaciones necesarias para preparar el dataset para análisis y modelado posterior.

### Normalización y escalado — ¿por qué?
Los algoritmos de ML (KMeans, SVM, regresión con regularización) son sensibles a la escala de las variables. Si `price` va de 0 a 1000 y `minimum_nights` de 1 a 365, la variable con mayor rango domina el cálculo de distancias. Escalar lleva todas las variables al mismo rango.

### LabelEncoder vs OneHotEncoder — ¿cuándo usar cada uno?
- **LabelEncoder:** para variables ordinales (hay un orden: bajo < medio < alto). Asigna un entero a cada categoría.
- **OneHotEncoder / get_dummies:** para variables nominales sin orden (room_type: Entire home, Private room...). Crea una columna binaria por cada categoría. Evita que el modelo interprete relaciones de orden inexistentes.

In [ ]:
# SDD: Preprocesado — Rúbrica: Técnicas de preprocesado (normalización, escalado, encoders)
from sklearn.preprocessing import MinMaxScaler, LabelEncoder

df_processed = df_clean.copy()

# Imputar nulos en reviews_per_month (0 si no tiene reviews)
df_processed['reviews_per_month'] = df_processed['reviews_per_month'].fillna(0)

# Parsear fechas
df_processed['last_review'] = pd.to_datetime(df_processed['last_review'], errors='coerce')

# OneHotEncoder para room_type
df_processed = pd.get_dummies(df_processed, columns=['room_type'], prefix='rt')

# LabelEncoder para city (para uso en modelos que lo requieran)
le = LabelEncoder()
df_processed['city_encoded'] = le.fit_transform(df_processed['city'])
print('Mapping ciudad → código:', dict(zip(le.classes_, le.transform(le.classes_))))

# Escalado MinMax para variables numéricas
scaler = MinMaxScaler()
cols_escalar = ['price', 'minimum_nights', 'number_of_reviews', 'reviews_per_month']
df_processed[[f'{c}_scaled' for c in cols_escalar]] = scaler.fit_transform(df_processed[cols_escalar])

print(f'\nDataset procesado: {df_processed.shape[0]:,} filas × {df_processed.shape[1]} columnas')
df_processed.head(2)

In [ ]:
# Guardar dataset procesado
import os
os.makedirs('data/processed', exist_ok=True)
df_processed.to_csv('data/processed/airbnb_clean.csv', index=False)
print('✅ Dataset procesado guardado en data/processed/airbnb_clean.csv')

---
## 8. Conclusiones del EDA

> **[SDD]** Esta celda debe completarse tras ejecutar el notebook completo con los datos reales.  
> Documentar aquí: hallazgos principales, hipótesis generadas, decisiones de limpieza tomadas, y próximos pasos.

### Hallazgos principales
- [ ] Distribución de precios: ...
- [ ] Ciudades con más alojamientos: ...
- [ ] Tipo de alojamiento más frecuente: ...
- [ ] Variables con mayor correlación con precio: ...
- [ ] Outliers detectados y decisión tomada: ...

### Hipótesis para validar en fases siguientes
1. ¿El precio medio difiere significativamente entre ciudades? → Test t / ANOVA
2. ¿Los alojamientos con más reviews tienen precios más bajos?
3. ¿Las noches mínimas altas están correlacionadas con menor disponibilidad?

### Próximos pasos
- [ ] Fase 4: Visualizaciones avanzadas (pairplot, heatmap 3D, segmentación por barrio)
- [ ] Fase 5: Dashboard en Looker Studio
- [ ] Fase 8: Tests estadísticos para validar hipótesis